# P143 — Calibrar el ruido a la sensibilidad en el análisis privado de datos

## 1. Título y paper

**Paper:** *Calibrating Noise to Sensitivity in Private Data Analysis*  
**Autoría:** Cynthia Dwork, Frank McSherry, Kobbi Nissim, Adam Smith  
**Año y venue:** 2006 · TCC 2006, 265–284  
**Nivel:** L3 · **Motor:** `privacidad_diferencial`  
**Ficha completa:** [`P143_privacidad_diferencial`](../../papers/foundational/P143_privacidad_diferencial/README.md)

**Hito:** Da una definición formal de privacidad que no depende de qué sepa el atacante, y un mecanismo concreto para cumplirla.

- [doi:10.1007/11681878_14](https://doi.org/10.1007/11681878_14)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La anonimización fallaba una y otra vez: cruzando datos supuestamente anónimos con otras fuentes se reidentificaba a personas. El problema de fondo es que cualquier definición basada en «quitar los identificadores» depende de qué más sepa quien ataca, y eso no se puede acotar.
2. Ejecutar una implementación mínima de la propuesta: Definir la privacidad como una propiedad del **mecanismo**: que la salida cambie poco —acotado por ε— cuando se añade o quita una persona. Y dar un mecanismo que la cumple: añadir ruido de Laplace calibrado a la sensibilidad de la consulta.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P134


## 4. Intuición

Dos consultas que solo difieren en una persona. La resta revela su dato con certeza. Publicar agregados exactos no protege a nadie — y ninguna anonimización arregla eso.


## 5. Concepto mínimo

```text
M es ε-diferencialmente privado ⟺ para todo par de bases D, D' que difieren
en UNA persona, y todo resultado S:

    P[M(D) ∈ S]  ≤  e^ε · P[M(D') ∈ S]

Mecanismo: sumar ruido de Laplace de escala (sensibilidad / ε)
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('privacidad_diferencial', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto acierta el ataque sin ruido?
2. ¿Y con ε pequeño?
3. ¿Qué se paga?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('privacidad_diferencial', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('privacidad_diferencial', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Sin ruido, el ataque de diferenciación acierta el **100 %**. Con ε = 0,1 cae al **48,2 %** — indistinguible de adivinar. Con ε = 10 sigue acertando el **99,3 %**. Y el precio: el error medio de la consulta pasa de 0,1 a 9,95 sobre un conteo de 249.


## 10. Comentario pedagógico

Ese ε = 10 es la trampa que hay que ver: **cumple la definición formal y no protege nada**. «Usamos privacidad diferencial» no es una afirmación verificable si no viene con el valor de ε, y elegirlo es una decisión de política, no técnica. Varios despliegues conocidos usan valores que buena parte de la comunidad considera demasiado laxos.


## 11. Error o anti-patrón deliberado

Anti-patrón: anunciar privacidad diferencial sin publicar ε.


In [ ]:
print('Con epsilon = 10 el ataque acierta el 99,3 % y la definicion se cumple igual.')
print('El parametro ES la garantia. Sin el, la afirmacion no dice nada.')
print('Y hay que declarar tambien cuantas consultas: el presupuesto se compone.')

## 12. Corrección

El compromiso completo:


In [ ]:
r = run_paper_lab('privacidad_diferencial', seed=3)['result']
print('sin privacidad:', r['sin_privacidad'])
for f in r['por_epsilon']:
    print(f)

## 13. Desafío guiado

Explica por qué la garantía no depende de qué sepa el atacante, y en qué se diferencia eso de la anonimización.


In [ ]:
r = run_paper_lab('privacidad_diferencial', seed=3)['result']
show(r)

## 14. Desafío autónomo

Busca un producto que anuncie privacidad diferencial y comprueba si publica su ε y su presupuesto de consultas.


## 15. Evidencia de aprendizaje

Guarda el valor —o la ausencia— y qué te dice.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P143_privacidad_diferencial/README.md) · evaluación formal: [`assessments/papers/P143_privacidad_diferencial.md`](../../assessments/papers/P143_privacidad_diferencial.md)


## 16. Cierre

Proteger los datos publicados es una mitad. La otra es no tener que recogerlos: eso es P146.


## 17. Conexión con el siguiente hito

- P146

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
